# Notebook 1 — Bronze & Silver
**Project:** Industrial Sensor Pipeline (NASA CMAPSS)

This notebook covers:
- Installing PySpark and downloading the NASA CMAPSS dataset
- **Bronze layer:** raw ingestion → Parquet
- **Silver layer:** validation, cleaning, status flags → Parquet

The NASA CMAPSS dataset simulates turbofan engine sensor readings over time.
Each row is one engine cycle with 21 sensor readings (temperature, pressure, rotation speed etc.).
This mirrors industrial IoT data from production environments.

## Setup

In [ ]:
# Install dependencies (only needed in Colab)
!pip install pyspark wget -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
import urllib.request
import zipfile
import os

# Start Spark session
spark = SparkSession.builder \
    .appName("IndustrialSensorPipeline") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## Download NASA CMAPSS Dataset

In [ ]:
# Download the CMAPSS dataset from NASA
url = "https://data.nasa.gov/api/views/nk8v-ckgx/rows.csv?accessType=DOWNLOAD"

# Alternative: use the commonly mirrored version from GitHub
# We'll use a reliable public mirror of the original CMAPSSData
import urllib.request

os.makedirs("data/raw", exist_ok=True)

# Download train_FD001.txt — engine degradation simulation dataset
base_url = "https://raw.githubusercontent.com/jordanh6/CMAPSS-dataset/main/"
files = ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"]

for f in files:
    urllib.request.urlretrieve(base_url + f, f"data/raw/{f}")
    print(f"Downloaded: {f}")

print("\nAll files downloaded.")

## Define Schema

CMAPSS columns:
- `engine_id`: unique engine unit
- `cycle`: time step (like a timestamp)
- `op_setting_1/2/3`: operational settings
- `sensor_01` to `sensor_21`: sensor measurements (temperature, pressure, rotation speed etc.)

In [ ]:
# Define explicit schema — good practice, avoids type inference errors
sensor_cols = [StructField(f"sensor_{i:02d}", DoubleType(), True) for i in range(1, 22)]

schema = StructType([
    StructField("engine_id",    IntegerType(), True),
    StructField("cycle",        IntegerType(), True),
    StructField("op_setting_1", DoubleType(),  True),
    StructField("op_setting_2", DoubleType(),  True),
    StructField("op_setting_3", DoubleType(),  True),
] + sensor_cols)

print(f"Schema defined: {len(schema.fields)} columns")

## Bronze Layer — Raw Ingestion

Bronze = append-only, no transformations. We store exactly what we received.
Adding `load_timestamp` for lineage tracking.

In [ ]:
# Read raw text file (space-separated, no header)
df_raw = spark.read \
    .schema(schema) \
    .option("sep", " ") \
    .option("header", "false") \
    .csv("data/raw/train_FD001.txt")

# Add load timestamp for lineage
df_bronze = df_raw.withColumn("load_timestamp", F.current_timestamp())

print(f"Rows ingested: {df_bronze.count():,}")
print(f"Engines: {df_bronze.select('engine_id').distinct().count()}")
print(f"Max cycle: {df_bronze.agg(F.max('cycle')).collect()[0][0]}")
df_bronze.printSchema()

In [ ]:
# Preview raw data
df_bronze.select("engine_id", "cycle", "op_setting_1", "sensor_01", "sensor_02", "sensor_03").show(10)

In [ ]:
# Write Bronze as Parquet — columnar format, much faster than CSV for analytics
os.makedirs("data/bronze", exist_ok=True)

df_bronze.write \
    .mode("overwrite") \
    .parquet("data/bronze/sensor_readings")

print("Bronze layer written to data/bronze/sensor_readings")

## Silver Layer — Validation & Cleaning

Silver = validated, flagged, analytics-ready structure.

Validation rules applied:
1. **Null check** — any row with nulls in key sensor columns is flagged
2. **Sensor range check** — sensor values must be > 0 (physical sensors can't read negative)
3. **Cycle order check** — cycles per engine must be sequential (no gaps > 1)
4. All invalid rows are **kept with a status flag** — same design decision as finance-data-pipeline

In [ ]:
# Read from Bronze
df_bronze = spark.read.parquet("data/bronze/sensor_readings")

# Key sensors to validate (subset — temperature, pressure, rotation)
key_sensors = ["sensor_02", "sensor_03", "sensor_04", "sensor_07", "sensor_11"]

# Rule 1: Null check on key sensors and required fields
null_condition = F.lit(False)
for col in ["engine_id", "cycle"] + key_sensors:
    null_condition = null_condition | F.col(col).isNull()

# Rule 2: Sensor values must be positive
negative_condition = F.lit(False)
for col in key_sensors:
    negative_condition = negative_condition | (F.col(col) <= 0)

# Assign status flags using when/otherwise — equivalent to CASE WHEN in SQL
df_silver = df_bronze.withColumn(
    "status",
    F.when(null_condition,     F.lit("invalid_null"))
     .when(negative_condition, F.lit("invalid_sensor_range"))
     .otherwise(               F.lit("valid"))
)

# Validation summary
print("=== Validation Summary ===")
df_silver.groupBy("status").count().orderBy("status").show()

valid_pct = df_silver.filter(F.col("status") == "valid").count() / df_silver.count() * 100
print(f"Valid rows: {valid_pct:.1f}%")

In [ ]:
# Add derived columns useful for analytics
# max_cycle per engine = total engine lifetime (needed for RUL calculation later)
from pyspark.sql.window import Window

window_engine = Window.partitionBy("engine_id")

df_silver = df_silver \
    .withColumn("max_cycle",      F.max("cycle").over(window_engine)) \
    .withColumn("cycle_pct",      F.round(F.col("cycle") / F.col("max_cycle"), 3)) \
    .withColumn("silver_timestamp", F.current_timestamp())

# Preview
df_silver.select(
    "engine_id", "cycle", "max_cycle", "cycle_pct",
    "sensor_02", "sensor_04", "status"
).show(10)

In [ ]:
# Write Silver as Parquet
os.makedirs("data/silver", exist_ok=True)

df_silver.write \
    .mode("overwrite") \
    .parquet("data/silver/sensor_readings_validated")

print("Silver layer written to data/silver/sensor_readings_validated")
print(f"Total rows: {df_silver.count():,}")
print(f"Valid rows: {df_silver.filter(F.col('status') == 'valid').count():,}")

## Summary

| Layer  | Format  | Rows | Notes |
|--------|---------|------|-------|
| Bronze | Parquet | all  | Raw + load_timestamp |
| Silver | Parquet | all  | Validated, status-flagged, cycle_pct added |

**Next:** Notebook 2 — Gold layer with Spark SQL analytics (rolling averages, sensor trends, degradation patterns)